# Diffusion figures for `blog.md`

Figures for the *Diffusion* section of `blog.md`. Everything is written to `figs/diffusion/`.
The forward-process figures use a real scan from `r2flow/data/go2w_sim`; the reverse-process
figures use a small DDPM trained here on a 2D toy (two moons, the same toy the rectified-flow
figures in `flow_matching.ipynb` use), which takes well under a minute.

Conventions follow the blog: $t=0$ is data, $t=T$ is noise, $\beta_t$ is the linear schedule of
Ho et al. (2020), $\alpha_t=1-\beta_t$ and $\bar\alpha_t=\prod_{s\le t}\alpha_s$.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from matplotlib.animation import FuncAnimation, PillowWriter

OUT = Path("../figs/diffusion")
OUT.mkdir(parents=True, exist_ok=True)

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# same palette as notebooks/flow_matching.ipynb
NOISE, DATA, CA, CB, INK = "#8a9099", "#c8102e", "#1f6fb2", "#e8890c", "#222222"
plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 100, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.labelsize": 11,
})


def save_png(fig, name):
    path = OUT / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    print(f"wrote {path}")


def save_gif(anim, name, fps=20, dpi=90):
    path = OUT / f"{name}.gif"
    anim.save(path, writer=PillowWriter(fps=fps), dpi=dpi)
    plt.close(anim._fig)
    print(f"wrote {path} ({path.stat().st_size / 1e6:.2f} MB)")


def with_holds(n_move, hold_start=10, hold_end=25):
    return np.concatenate([np.zeros(hold_start), np.linspace(0, 1, n_move), np.ones(hold_end)])


# ---------------------------------------------------------------- DDPM schedule
T = 1000
betas = np.linspace(1e-4, 0.02, T)          # Ho et al. (2020), linear schedule
alphas = 1.0 - betas
abars = np.cumprod(alphas)

## 0. Loading a scan

Projection onto the sensor's ray grid, and the log-scale encoding in $[-1,1]$ the models train
on. This is the same nearest-cell spherical projection `r2flow/data/go2w_sim/go2w_sim.py` uses,
rewritten without the `numba`/`datasets` dependencies so this notebook stands alone.

In [2]:
import numpy as np
from pathlib import Path

H, W = 40, 512
H_UP, H_DOWN = 29.50, -29.70
MIN_DEPTH, MAX_DEPTH = 0.5, 80.0
LOG_MAX = np.log2(MAX_DEPTH + 1.0)


def load_range_image(path):
    """-> (depth HxW in metres, mask HxW). Nearest-cell spherical projection, closest wins."""
    pts = np.fromfile(path, dtype=np.float32).reshape(-1, 4)
    xyz = pts[:, :3]
    depth = np.linalg.norm(xyz, axis=1)
    keep = (depth >= MIN_DEPTH) & (depth <= MAX_DEPTH)
    xyz, depth = xyz[keep], depth[keep]

    elev = np.degrees(np.arcsin(xyz[:, 2] / np.maximum(depth, 1e-9)))
    gh = np.rint((H_UP - elev) / ((H_UP - H_DOWN) / H)).astype(np.int32)
    azim = np.degrees(np.arctan2(xyz[:, 1], xyz[:, 0]))
    gw = np.rint((180.0 - azim) / (360.0 / W)).astype(np.int32) % W

    inside = (gh >= 0) & (gh < H)
    gh, gw, depth = gh[inside], gw[inside], depth[inside]

    img = np.zeros((H, W), np.float32)
    order = np.argsort(-depth)                 # far first, so the closest point wins
    img[gh[order], gw[order]] = depth[order]
    return img, (img > 0).astype(np.float32)


def to_normalized(depth):
    """metres -> the log-scale code in [-1, 1] the models actually train on."""
    return np.clip(np.log2(depth + 1.0) / LOG_MAX, 0, 1) * 2 - 1


def to_metric(x):
    return np.exp2(np.clip((x + 1) / 2, 0, 1) * LOG_MAX) - 1.0


def ray_grid():
    """Unit ray direction for every cell, (H, W, 3)."""
    elev = np.radians(H_UP - (np.arange(H) + 0.5) * (H_UP - H_DOWN) / H)[:, None]
    azim = np.radians(180.0 - (np.arange(W) + 0.5) * (360.0 / W))[None, :]
    return np.stack([np.cos(elev) * np.cos(azim), np.cos(elev) * np.sin(azim),
                     np.sin(elev) * np.ones_like(azim)], -1)


RAYS = ray_grid()


def to_points(depth, mask=None):
    """Range image -> (N, 3) point cloud."""
    sel = np.ones_like(depth, bool) if mask is None else mask > 0.5
    return RAYS[sel] * depth[sel][:, None]


def scan_paths(split="go2w_sim"):
    d = Path(f"../r2flow/data/{split}/dataset/velodyne_points/data")
    return sorted(d.glob("*.bin"))

## 1. The two processes (section "Diffusion")

The forward process walks a real scan off the manifold; the reverse process walks a noise sample
back onto it, landing somewhere else. The left leg is the forward noising of one real scan; the
right leg is the exact posterior chain $q(x_{t-1}\mid x_t,x_0)$ aimed at a *different* real scan,
which is the trajectory the learned $p_\theta$ has to imitate without being shown its target.

In [3]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Ellipse, FancyArrowPatch


BEV_LIM = 8.5
FWD = [0, 30, 120]        # forward waypoints, then the noise cloud at t = T
REV = [120, 30, 0]        # reverse waypoints after the cloud

paths = scan_paths("go2w_sim")
xA = to_normalized(load_range_image(paths[20])[0])    # the scan the forward leg starts from
xB = to_normalized(load_range_image(paths[180])[0])   # the scan the reverse leg lands on

# forward leg: the closed form, one draw of eps held fixed
eps = rng.standard_normal(xA.shape)
fwd = {t: (xA if t == 0 else np.sqrt(abars[t - 1]) * xA + np.sqrt(1 - abars[t - 1]) * eps) for t in FWD}

# reverse leg: the exact x_0-conditioned posterior chain, aimed at xB
ab_prev = np.concatenate([[1.0], abars[:-1]])
c0 = np.sqrt(ab_prev) * betas / (1 - abars)
ct = np.sqrt(alphas) * (1 - ab_prev) / (1 - abars)
bt = (1 - ab_prev) / (1 - abars) * betas

x = rng.standard_normal(xB.shape)
noise_img = x.copy()
rev = {}
for t in range(T, 0, -1):
    x = c0[t - 1] * xB + ct[t - 1] * x + np.sqrt(bt[t - 1]) * rng.standard_normal(xB.shape)
    if t - 1 in REV:
        rev[t - 1] = x.copy()


def draw_bev(ax, xn, edge):
    pts = to_points(to_metric(xn))
    r = np.linalg.norm(pts, axis=1)
    k = (r > 0.5) & (np.abs(pts[:, 0]) < BEV_LIM) & (np.abs(pts[:, 1]) < BEV_LIM)
    ax.scatter(pts[k, 1], pts[k, 0], s=1.5, c=r[k], cmap="turbo", vmin=0, vmax=16,
               linewidths=0, alpha=0.9, rasterized=True)
    ax.set_xlim(-BEV_LIM, BEV_LIM); ax.set_ylim(-BEV_LIM, BEV_LIM)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_facecolor("#fbfbfc")
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_color(edge); sp.set_linewidth(1.6)


fig = plt.figure(figsize=(13.0, 7.2))
bg = fig.add_axes([0, 0, 1, 1]); bg.set_xlim(0, 1); bg.set_ylim(0, 1); bg.axis("off")

# the seven waypoints, on an ellipse from the manifold up through the noise and back down
ang = np.radians(np.linspace(180, 0, 7))
cx, cy = 0.5 + 0.38 * np.cos(ang), 0.30 + 0.50 * np.sin(ang)
TW, TH = 1.45 / 13.0, 1.45 / 7.2   # thumbnail size in figure coords

# the data manifold
mx = np.linspace(0.02, 0.98, 400)
my = 0.105 + 0.05 * np.sin(np.pi * mx)
bg.plot(mx, my, color=INK, lw=2.4, zorder=2)
bg.text(0.5, 0.035, "the manifold of real range images", color=INK, fontsize=11.5, ha="center")

# the prior
bg.add_patch(Ellipse((cx[3], cy[3]), 0.265, 0.245, color=NOISE, alpha=0.16, lw=0, zorder=1))
bg.text(cx[3], cy[3] + 0.168, r"$x_T\sim\mathcal{N}(0,I)$", color=INK, fontsize=12.5, ha="center")

labels = [r"$x_0$" + "\na real scan", "$t=30$", "$t=120$", f"$t=T={T}$", "$t=120$", "$t=30$",
          r"$\hat x_0$" + "\nback on\nthe manifold"]
edges = [DATA, CA, CA, NOISE, DATA, DATA, DATA]
imgs = [fwd[0], fwd[30], fwd[120], noise_img, rev[120], rev[30], rev[0]]

for i in range(7):
    ax = fig.add_axes([cx[i] - TW / 2, cy[i] - TH / 2, TW, TH])
    draw_bev(ax, imgs[i], edges[i])
    if i == 0:      # the two endpoints get their label to the outside
        bg.text(cx[i] - TW / 2 - 0.012, cy[i], labels[i], color=INK, fontsize=10.5,
                ha="right", va="center")
    elif i == 6:
        bg.text(cx[i] + TW / 2 + 0.012, cy[i], labels[i], color=INK, fontsize=10.5,
                ha="left", va="center")
    else:
        bg.text(cx[i], cy[i] + TH / 2 + 0.012, labels[i], color=INK, fontsize=10.5,
                ha="center", va="bottom")

# anchor the two endpoints to the manifold
for i in (0, 6):
    ym = 0.105 + 0.05 * np.sin(np.pi * cx[i])
    bg.plot([cx[i]], [ym], "o", color=DATA, ms=9, zorder=4)
    bg.plot([cx[i], cx[i]], [ym, cy[i] - TH / 2], color=DATA, lw=1.6, zorder=5)

for i in range(6):
    col = CA if i < 3 else DATA
    bg.add_patch(FancyArrowPatch((cx[i], cy[i]), (cx[i + 1], cy[i + 1]),
                                 connectionstyle="arc3,rad=-0.13", color=col, lw=2.0,
                                 arrowstyle="-|>", mutation_scale=18,
                                 shrinkA=42, shrinkB=42, zorder=3))

bg.text(0.015, 0.88, "forward process  $q(x_t\\mid x_{t-1})$\n"
        "add noise; the sample\nleaves the manifold", color=CA, fontsize=11.5, va="top")
bg.text(0.985, 0.88, "reverse process  $p_\\theta(x_{t-1}\\mid x_t)$\n"
        "learned; walks the sample\nback onto the manifold", color=DATA, fontsize=11.5,
        va="top", ha="right")

save_png(fig, "diffusion_manifold")
plt.close(fig)

wrote ../figs/diffusion/diffusion_manifold.png


## 2. Brownian motion vs. the rescaled step (section "Forward Process")

Why $dX_t=\sqrt{2D}\,dW_t$ cannot be the forward process: its variance grows without bound.
Rescaling by $\sqrt{1-\beta_t}$ at every step pulls the mass back in as fast as the noise
spreads it out, so the marginal settles on $\mathcal{N}(0,I)$.

In [4]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap


# a stand-in for p_data: three sharp modes, so "structure" is visible
MU = np.array([-2.2, 0.3, 2.6])
SD = np.array([0.35, 0.50, 0.30])
WT = np.array([0.35, 0.30, 0.35])
VAR0 = float((WT * (SD ** 2 + MU ** 2)).sum() - (WT * MU).sum() ** 2)

xs = np.linspace(-9, 9, 900)
FRACS = [0.0, 0.02, 0.08, 0.25, 1.0]
cmap = LinearSegmentedColormap.from_list("t", [DATA, CA, NOISE])
D2 = 25.0  # 2D * t_max, so pure Brownian ends at std 5


def mixture(mu, var):
    return (WT[:, None] * np.exp(-0.5 * (xs[None] - mu[:, None]) ** 2 / var[:, None])
            / np.sqrt(2 * np.pi * var[:, None])).sum(0)


def ve(s):   # dX = sqrt(2D) dW  ->  N(x0, 2Dt)
    return mixture(MU, SD ** 2 + D2 * s)


def vp(s):   # x_t = sqrt(abar) x0 + sqrt(1-abar) eps
    ab = 1.0 if s == 0 else abars[int(round(s * T)) - 1]
    return mixture(np.sqrt(ab) * MU, ab * SD ** 2 + (1 - ab))


fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.9))

for ax, fn, title in (
    (axes[0], ve, r"(a) pure Brownian motion: $x_t=x_0+\sqrt{2Dt}\,\epsilon$"),
    (axes[1], vp, r"(b) the rescaled step: $x_t=\sqrt{1-\beta_t}\,x_{t-1}+\sqrt{\beta_t}\,\epsilon_t$"),
):
    ax.plot(xs, np.exp(-0.5 * xs ** 2) / np.sqrt(2 * np.pi), color=INK, ls="--", lw=1.4,
            label=r"$\mathcal{N}(0,1)$", zorder=1)
    for s in FRACS:
        lab = {0.0: "start: $p_{\\mathrm{data}}$", 1.0: "end of the process"}.get(s)
        ax.plot(xs, fn(s), color=cmap(s ** 0.4), lw=2.0, zorder=2, label=lab)
    ax.set_xlim(-9, 9); ax.set_ylim(0, 0.78)
    ax.set_xlabel("$x$"); ax.set_title(title, loc="left", fontsize=11)
    ax.legend(frameon=False, loc="upper left", fontsize=9)

axes[0].set_ylabel("density")
axes[0].annotate("the spread grows without\nbound, so it never settles\non $\\mathcal{N}(0,1)$", (6.2, 0.055),
                 xytext=(1.2, 0.74), fontsize=9.5, color=NOISE, va="top",
                 arrowprops=dict(arrowstyle="->", color=NOISE, lw=1))
axes[1].annotate("shrinking $x_{t-1}$ pulls the\nmass back in as fast as the\nnoise spreads it out", (0.9, 0.36),
                 xytext=(2.6, 0.74), fontsize=9.5, color=NOISE, va="top",
                 arrowprops=dict(arrowstyle="->", color=NOISE, lw=1))

ax = axes[2]
ss = np.linspace(0, 1, 400)
ax.plot(ss, VAR0 + D2 * ss, color=NOISE, lw=2.2, label="Brownian (variance explodes)")
ab = np.concatenate([[1.0], abars])[np.round(ss * T).astype(int)]
ax.plot(ss, ab * VAR0 + (1 - ab), color=DATA, lw=2.2, label="rescaled (variance preserved)")
ax.axhline(1.0, color=INK, ls="--", lw=1.2)
ax.text(0.52, 1.9, r"$\mathrm{Var}=1$", fontsize=9.5, color=INK)
ax.set_xlabel("progress through the process"); ax.set_ylabel(r"$\mathrm{Var}[x_t]$")
ax.set_xlim(0, 1); ax.set_ylim(0, 27)
ax.set_title("(c) where the variance ends up", loc="left", fontsize=11)
ax.legend(frameon=False, loc="upper left", fontsize=9.5)

fig.tight_layout()
save_png(fig, "brownian_vs_vp")
plt.close(fig)

wrote ../figs/diffusion/brownian_vs_vp.png


## 3. The schedule and what it compounds into

$\beta_t$ is tiny at every single step, but $\bar\alpha_t$ is a product of a thousand of them.

In [5]:
import matplotlib.pyplot as plt
import numpy as np


MARKS = [25, 75, 150, 400, 1000]
ts = np.arange(1, T + 1)

fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.6))

ax = axes[0]
ax.plot(ts, betas, color=INK, lw=2)
ax.set_xlabel("$t$"); ax.set_ylabel(r"$\beta_t$")
ax.set_title(r"(a) the schedule: $\beta_t$ linear from $10^{-4}$ to $0.02$", loc="left", fontsize=11)
ax.set_xlim(0, T); ax.set_ylim(0, 0.022)
ax.text(40, 0.0185, "each single step\nadds very little noise", fontsize=9, color=NOISE, va="top")

ax = axes[1]
ax.plot(ts, np.sqrt(abars), color=DATA, lw=2.2, label=r"$\sqrt{\bar\alpha_t}$  (signal)")
ax.plot(ts, np.sqrt(1 - abars), color=CA, lw=2.2, label=r"$\sqrt{1-\bar\alpha_t}$  (noise)")
for m in MARKS:
    ax.plot([m, m], [0, np.sqrt(abars[m - 1])], color=NOISE, ls=":", lw=1)
    ax.plot([m], [np.sqrt(abars[m - 1])], "o", color=DATA, ms=4.5, zorder=4)
    ax.plot([m], [np.sqrt(1 - abars[m - 1])], "o", color=CA, ms=4.5, zorder=4)
ax.set_xlabel("$t$"); ax.set_xlim(0, T); ax.set_ylim(0, 1.05)
ax.set_title(r"(b) but they compound: $x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon$",
             loc="left", fontsize=11)
ax.legend(frameon=False, loc="center right", fontsize=10)

ax = axes[2]
snr = abars / (1 - abars)
ax.plot(ts, snr, color=CB, lw=2.2)
ax.set_yscale("log"); ax.set_xlim(0, T)
ax.axhline(1.0, color=NOISE, ls="--", lw=1)
cross = int(np.argmin(np.abs(np.log(snr)))) + 1
ax.plot([cross], [1.0], "o", color=INK, ms=6, zorder=5)
ax.annotate(f"signal = noise\nat $t\\approx{cross}$", (cross, 1.0), textcoords="offset points",
            xytext=(18, 26), fontsize=9.5, color=INK,
            arrowprops=dict(arrowstyle="->", color=INK, lw=1))
ax.set_xlabel("$t$"); ax.set_ylabel(r"$\mathrm{SNR}_t=\bar\alpha_t/(1-\bar\alpha_t)$")
ax.set_title("(c) signal-to-noise ratio, log scale", loc="left", fontsize=11)

fig.tight_layout()
save_png(fig, "noise_schedule")
plt.close(fig)

wrote ../figs/diffusion/noise_schedule.png


## 4. The forward process on a real scan (Eq. 2)

The same scan, noised with the closed form
$x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$, as a range image and as the point
cloud it decodes to. One draw of $\epsilon$ is held fixed across $t$ so the rows are comparable.

In [6]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec


STEPS = [0, 25, 75, 150, 400, 1000]
LIM = 20.0

depth0, _ = load_range_image(scan_paths("go2w_sim")[100])
x0 = to_normalized(depth0)
eps = rng.standard_normal(x0.shape)

fig = plt.figure(figsize=(13.0, 8.2))
gs = GridSpec(2, 1, height_ratios=[1.35, 1.0], hspace=0.2, figure=fig)
gs_img = gs[0].subgridspec(len(STEPS), 1, hspace=0.6)
gs_bev = gs[1].subgridspec(1, len(STEPS), wspace=0.06)

for j, t in enumerate(STEPS):
    ab = 1.0 if t == 0 else abars[t - 1]
    xt = np.sqrt(ab) * x0 + np.sqrt(1 - ab) * eps

    ax = fig.add_subplot(gs_img[j])
    ax.imshow(xt, cmap="turbo", vmin=-1.8, vmax=1.8, aspect="auto", interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True); s.set_color("#cccccc")
    ax.set_ylabel(f"$t={t}$", rotation=0, ha="right", va="center", labelpad=10)
    ax.set_title(rf"$\sqrt{{\bar\alpha_t}}={np.sqrt(ab):.3f}$"
                 rf"      $\sqrt{{1-\bar\alpha_t}}={np.sqrt(1 - ab):.3f}$",
                 loc="left", fontsize=9.5, color=INK, pad=3)

    pts = to_points(to_metric(xt))
    r = np.linalg.norm(pts, axis=1)
    keep = (r > 0.5) & (np.abs(pts[:, 0]) < LIM) & (np.abs(pts[:, 1]) < LIM)
    axb = fig.add_subplot(gs_bev[j])
    axb.scatter(pts[keep, 1], pts[keep, 0], s=0.6, c=r[keep], cmap="turbo",
                vmin=0, vmax=35, linewidths=0, alpha=0.85, rasterized=True)
    axb.set_xlim(-LIM, LIM); axb.set_ylim(-LIM, LIM); axb.set_aspect("equal")
    axb.set_xticks([]); axb.set_yticks([])
    for s in axb.spines.values():
        s.set_color("#cccccc")
    axb.set_title(f"$t={t}$", fontsize=10.5, color=INK, pad=4)

fig.text(0.5, 0.94, r"the range image $x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon$,"
                    " one draw of $\\epsilon$ held fixed across $t$", ha="center", fontsize=12.5, color=INK)
fig.text(0.5, 0.445, r"the same $x_t$ decoded back to a point cloud (bird's eye view, $\pm$20 m)",
         ha="center", fontsize=12.5, color=INK)
save_png(fig, "forward_lidar")
plt.close(fig)

wrote ../figs/diffusion/forward_lidar.png


## 5. Why $q(x_t)$ is intractable (section "Reverse Process")

$q(x_t\mid x_0)$ is one Gaussian. $q(x_t)=\int q(x_t\mid x_0)p_{\mathrm{data}}(x_0)\,dx_0$
needs every $x_0$ at once, and for range images that integral runs over $\mathbb{R}^{40\times512}$.

In [7]:
import matplotlib.pyplot as plt
import numpy as np


MU = np.array([-2.2, 0.3, 2.6])
SD = np.array([0.35, 0.50, 0.30])
WT = np.array([0.35, 0.30, 0.35])
TSHOW = 300
AB = abars[TSHOW - 1]
xs = np.linspace(-6, 6, 900)


def gauss(m, v):
    return np.exp(-0.5 * (xs - m) ** 2 / v) / np.sqrt(2 * np.pi * v)


p_data = sum(w * gauss(m, s ** 2) for w, m, s in zip(WT, MU, SD))
comp = rng.choice(3, size=70, p=WT)
samples = MU[comp] + SD[comp] * rng.standard_normal(70)
picks = np.array([-2.35, 0.45, 2.55])
COLS = (CA, CB, "#2e8b57")

fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.0), sharey=True, sharex=True)

ax = axes[0]
ax.fill_between(xs, p_data, color=DATA, alpha=0.18, lw=0)
ax.plot(xs, p_data, color=DATA, lw=2.2)
ax.plot(picks, np.zeros(3), "*", color=INK, ms=14, clip_on=False, zorder=5)
ax.set_title(r"(a) $p_{\mathrm{data}}(x_0)$ — unknown", loc="left", fontsize=11)
ax.set_xlabel("$x_0$"); ax.set_ylabel("density")
ax.text(-5.8, 0.76, "we never get the curve,\nonly samples from it (stars)",
        fontsize=9.5, color=NOISE, va="top")

ax = axes[1]
for x0, c in zip(picks, COLS):
    m = np.sqrt(AB) * x0
    ax.plot(xs, gauss(m, 1 - AB), color=c, lw=2.2)
    ax.plot([m, m], [0, gauss(m, 1 - AB).max()], color=c, ls=":", lw=1)
    ax.plot([x0], [0], "*", color=INK, ms=13, clip_on=False, zorder=5)
ax.annotate(r"the mean is pulled in" "\n" r"from $x_0$ to $\sqrt{\bar\alpha_t}x_0$",
            (np.sqrt(AB) * picks[0], 0.04), xytext=(-5.85, 0.33), fontsize=9, color=CA,
            ha="left", va="top", arrowprops=dict(arrowstyle="->", color=CA, lw=1))
ax.set_title(rf"(b) $q(x_t\mid x_0)$ at $t={TSHOW}$ — closed form", loc="left", fontsize=11)
ax.set_xlabel("$x_t$")
ax.text(-5.8, 0.76, r"$\mathcal{N}\!\left(\sqrt{\bar\alpha_t}x_0,(1-\bar\alpha_t)I\right)$:"
        "\none Gaussian per starting point", fontsize=9.5, color=NOISE, va="top")

ax = axes[2]
for x0 in samples:
    ax.plot(xs, gauss(np.sqrt(AB) * x0, 1 - AB), color=NOISE, lw=0.7, alpha=0.22)
q_t = sum(w * gauss(np.sqrt(AB) * m, AB * s ** 2 + (1 - AB)) for w, m, s in zip(WT, MU, SD))
ax.fill_between(xs, q_t, color=CA, alpha=0.2, lw=0)
ax.plot(xs, q_t, color=CA, lw=2.6)
ax.set_title(r"(c) $q(x_t)=\int q(x_t\mid x_0)\,p_{\mathrm{data}}(x_0)\,dx_0$", loc="left", fontsize=11)
ax.set_xlabel("$x_t$")
ax.text(-5.8, 0.76, "the average of one bump per $x_0$.\nFor range images that integral\nruns over $\\mathbb{R}^{40\\times 512}$.",
        fontsize=9.5, color=NOISE, va="top")

axes[0].set_ylim(0, 0.84); axes[0].set_xlim(-6, 6)
fig.tight_layout()
save_png(fig, "intractable_marginal")
plt.close(fig)

wrote ../figs/diffusion/intractable_marginal.png


## 6. The reverse posterior $q(x_{t-1}\mid x_t,x_0)$

Its mean is an affine blend of $x_t$ and $x_0$. Away from the last few steps the weight on $x_0$
is a fraction of a percent, so one reverse step barely moves. Running the whole chain with the
true $x_0$ in hand recovers $x_0$ exactly, because $\tilde\beta_1=0$.

In [8]:
import matplotlib.pyplot as plt
import numpy as np


ab_prev = np.concatenate([[1.0], abars[:-1]])
c0 = np.sqrt(ab_prev) * betas / (1 - abars)              # weight on x_0
ct = np.sqrt(alphas) * (1 - ab_prev) / (1 - abars)       # weight on x_t
bt = (1 - ab_prev) / (1 - abars) * betas                 # posterior variance
ts = np.arange(1, T + 1)

fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.0))

ax = axes[0]
ax.plot(ts, c0, color=DATA, lw=2.3,
        label=r"on $x_0$:  $\dfrac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}$")
ax.plot(ts, ct, color=CA, lw=2.3,
        label=r"on $x_t$:  $\dfrac{\sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}$")
ax.set_xscale("log"); ax.set_xlim(1, T); ax.set_ylim(0, 1.1)
ax.set_xlabel("$t$ (log scale)"); ax.set_ylabel(r"weight in $\tilde\mu_t(x_t,x_0)$")
ax.set_title(r"(a) $\tilde\mu_t$ is a blend of $x_t$ and $x_0$", loc="left", fontsize=11)
ax.legend(frameon=False, loc="center left", fontsize=11, bbox_to_anchor=(0.03, 0.42))
for t in (100, 900):
    ax.plot([t], [c0[t - 1]], "o", color=DATA, ms=5, zorder=5)
    ax.annotate(f"$t={t}$:  {c0[t-1]*100:.2f}% of the\nway to $x_0$", (t, c0[t - 1]),
                textcoords="offset points", xytext=(-16, 26), fontsize=9, color=DATA, ha="center",
                arrowprops=dict(arrowstyle="->", color=DATA, lw=1))
ax.text(22, 0.80, "away from the last few steps,\none reverse step is almost entirely\n\"stay where you are\"",
        fontsize=9.5, color=NOISE, va="top")

ax = axes[1]
ax.plot(ts, np.sqrt(betas), color=NOISE, lw=2.6, label=r"forward step:  $\sqrt{\beta_t}$")
ax.plot(ts, np.sqrt(bt), color=CB, lw=2.0, ls="--", label=r"posterior given $x_0$:  $\sqrt{\tilde\beta_t}$")
ax.set_xscale("log"); ax.set_xlim(1, T); ax.set_ylim(0, 0.16)
ax.set_xlabel("$t$ (log scale)"); ax.set_ylabel("standard deviation")
ax.set_title(r"(b) knowing $x_0$ only narrows the step near $t=0$", loc="left", fontsize=11)
ax.legend(frameon=False, loc="upper left", fontsize=10)
ax.annotate(r"$\tilde\beta_1=0$: at the last step $x_0$ fixes the answer",
            (1.0, 0.0), xytext=(1.6, 0.052), fontsize=9.5, color=CB,
            arrowprops=dict(arrowstyle="->", color=CB, lw=1))

fig.tight_layout()
save_png(fig, "posterior_weights")
plt.close(fig)

wrote ../figs/diffusion/posterior_weights.png


In [9]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec


SHOW = [1000, 700, 400, 200, 50, 0]

depth0, _ = load_range_image(scan_paths("go2w_sim")[100])
x0 = to_normalized(depth0)

ab_prev = np.concatenate([[1.0], abars[:-1]])
c0 = np.sqrt(ab_prev) * betas / (1 - abars)
ct = np.sqrt(alphas) * (1 - ab_prev) / (1 - abars)
bt = (1 - ab_prev) / (1 - abars) * betas

# x_T ~ N(0, I), then x_{t-1} ~ q(x_{t-1} | x_t, x_0) all the way down
x = rng.standard_normal(x0.shape)
frames = {T: x.copy()}
err = np.zeros(T + 1)
err[T] = np.mean((x - x0) ** 2)
for t in range(T, 0, -1):
    mu = c0[t - 1] * x0 + ct[t - 1] * x
    x = mu + np.sqrt(bt[t - 1]) * rng.standard_normal(x0.shape)
    err[t - 1] = np.mean((x - x0) ** 2)
    if t - 1 in SHOW:
        frames[t - 1] = x.copy()

fig = plt.figure(figsize=(13.0, 7.0))
gs = GridSpec(2, 1, height_ratios=[1.45, 1.0], hspace=0.26, figure=fig)
gs_img = gs[0].subgridspec(len(SHOW), 1, hspace=0.55)

for j, t in enumerate(SHOW):
    ax = fig.add_subplot(gs_img[j])
    ax.imshow(frames[t], cmap="turbo", vmin=-1.8, vmax=1.8, aspect="auto", interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True); s.set_color("#cccccc")
    ax.set_ylabel(f"$t={t}$", rotation=0, ha="right", va="center", labelpad=10)
    if t == 0:
        ax.set_title(r"$x_0$ recovered exactly ($\tilde\beta_1=0$)", loc="left", fontsize=9.5, color=DATA, pad=3)
    elif t == T:
        ax.set_title(r"started from $x_T\sim\mathcal{N}(0,I)$", loc="left", fontsize=9.5, color=INK, pad=3)

ax = fig.add_subplot(gs[1])
ax.plot(np.arange(T + 1), err, color=CA, lw=2.2)
ax.invert_xaxis()
ax.set_xlabel("$t$  (the chain runs right to left)")
ax.set_ylabel(r"mean squared error to $x_0$")
ax.set_xlim(T, 0); ax.set_ylim(0, err.max() * 1.08)
for t in SHOW:
    ax.plot([t], [err[t]], "o", color=INK, ms=5, zorder=5)
ax.set_title(r"(b) each draw from $q(x_{t-1}\mid x_t,x_0)$ moves a little closer to $x_0$",
             loc="left", fontsize=11)
ax.text(880, err.max() * 0.35, "almost nothing happens for the first\nfew hundred steps: the weight on $x_0$\nis still well under a percent",
        fontsize=9.5, color=NOISE, va="top")

fig.text(0.5, 0.955, r"(a) sampling the chain $x_{t-1}\sim q(x_{t-1}\mid x_t,x_0)$ with the true $x_0$ in hand",
         ha="center", fontsize=12.5, color=INK)
save_png(fig, "posterior_chain_lidar")
plt.close(fig)
print("final MSE", err[0])

wrote ../figs/diffusion/posterior_chain_lidar.png
final MSE 0.0


## 7. A trained reverse process (section "Putting It All Together")

A small $\epsilon_\theta(x_t,t)$ trained with $\mathcal{L}_{\mathrm{simple}}$ on two moons, then
sampled ancestrally. The second panel shows the network's implied guess at the clean sample,
$\hat x_0=(x_t-\sqrt{1-\bar\alpha_t}\,\epsilon_\theta)/\sqrt{\bar\alpha_t}$: early on it is the
blurred average of the data, and it only resolves into two moons near the end.

In [10]:
import numpy as np
import torch
import torch.nn as nn


ab_t = torch.tensor(abars, dtype=torch.float32, device=DEVICE)


def moons(B, noise=0.06):
    s = torch.rand(B, device=DEVICE) * np.pi
    up = torch.rand(B, device=DEVICE) < 0.5
    x = torch.where(up, torch.cos(s), 1 - torch.cos(s))
    y = torch.where(up, torch.sin(s), 0.5 - torch.sin(s))
    pts = torch.stack([x - 0.5, y - 0.25], 1) * 2.0
    return pts + noise * torch.randn_like(pts)


class EpsMLP(nn.Module):
    """eps_theta(x_t, t), with a sinusoidal embedding of the discrete timestep."""

    def __init__(self, d=2, h=256, depth=4, emb=64):
        super().__init__()
        self.emb = emb
        freqs = torch.exp(torch.linspace(0, np.log(1000.0), emb // 2))
        self.register_buffer("freqs", freqs)
        layers, w = [], d + emb
        for _ in range(depth):
            layers += [nn.Linear(w, h), nn.SiLU()]
            w = h
        self.net = nn.Sequential(*layers, nn.Linear(h, d))

    def forward(self, t, x):
        a = t[:, None].float() / T * self.freqs[None]
        return self.net(torch.cat([x, torch.sin(a), torch.cos(a)], 1))


def train_ddpm(model, sample_x0, steps=12000, B=1024, lr=2e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    losses = []
    for s in range(steps):
        x0 = sample_x0(B)
        t = torch.randint(1, T + 1, (B,), device=DEVICE)
        eps = torch.randn_like(x0)
        ab = ab_t[t - 1][:, None]
        xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
        loss = ((model(t, xt) - eps) ** 2).sum(1).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        losses.append(loss.item())
    return np.array(losses)


@torch.no_grad()
def sample(model, xT, schedule, eta=1.0, keep=None):
    """Ancestral sampling along `schedule` (descending timesteps, ending at 0).

    eta=1 on the full 1..T schedule is exactly the DDPM sampler; the same formula
    with a strided schedule is how you take fewer, larger steps.
    """
    x = xT
    out = {}
    for i, s in enumerate(schedule):
        r = schedule[i + 1] if i + 1 < len(schedule) else 0
        ab_s = ab_t[s - 1]
        ab_r = ab_t[r - 1] if r > 0 else torch.tensor(1.0, device=DEVICE)
        eps = model(torch.full((x.shape[0],), s, device=DEVICE, dtype=torch.long), x)
        x0h = (x - (1 - ab_s).sqrt() * eps) / ab_s.sqrt()
        sig = eta * ((1 - ab_r) / (1 - ab_s)).sqrt() * (1 - ab_s / ab_r).sqrt()
        x = ab_r.sqrt() * x0h + (1 - ab_r - sig ** 2).clamp(min=0).sqrt() * eps
        if r > 0:
            x = x + sig * torch.randn_like(x)
        if keep is not None and r in keep:
            out[r] = x.cpu().numpy().copy()
    return (x, out) if keep is not None else x


def strided(K):
    """K evenly spaced timesteps from T down to 1."""
    return np.unique(np.round(np.linspace(1, T, K)).astype(int))[::-1].tolist()


def energy_distance(a, b):
    def pd(u, v):
        return np.linalg.norm(u[:, None] - v[None], axis=-1).mean()
    return 2 * pd(a, b) - pd(a, a) - pd(b, b)

In [11]:
torch.manual_seed(SEED)
net = EpsMLP().to(DEVICE)
losses = train_ddpm(net, moons)
print(f"final loss {losses[-200:].mean():.4f}")

final loss 0.4996


In [12]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.animation import FuncAnimation


N = 500
torch.manual_seed(7)
x = torch.randn(N, 2, device=DEVICE)
traj, guess = [x.cpu().numpy().copy()], []

with torch.no_grad():
    for s in range(T, 0, -1):
        ab_s = ab_t[s - 1]
        ab_r = ab_t[s - 2] if s > 1 else torch.tensor(1.0, device=DEVICE)
        tt = torch.full((N,), s, device=DEVICE, dtype=torch.long)
        eps = net(tt, x)
        x0h = (x - (1 - ab_s).sqrt() * eps) / ab_s.sqrt()
        guess.append(x0h.cpu().numpy().copy())
        sig = ((1 - ab_r) / (1 - ab_s)).sqrt() * (1 - ab_s / ab_r).sqrt()
        x = ab_r.sqrt() * x0h + (1 - ab_r - sig ** 2).clamp(min=0).sqrt() * eps
        if s > 1:
            x = x + sig * torch.randn_like(x)
        traj.append(x.cpu().numpy().copy())

traj = np.stack(traj)            # (T+1, N, 2), index k = after k steps, timestep T-k
guess = np.stack(guess)          # (T, N, 2), guess made at timestep T-k
data_bg = moons(4000).cpu().numpy()

NF = 70
ks = np.unique(np.round(T * (1 - (1 - np.linspace(0, 1, NF)) ** 2.5)).astype(int))
ks = np.concatenate([np.full(8, ks[0]), ks, np.full(22, ks[-1])])

fig, axes = plt.subplots(1, 2, figsize=(9.6, 5.0))
for ax in axes:
    ax.scatter(*data_bg.T, s=2, color=DATA, alpha=0.10, zorder=1)
    ax.set_xlim(-3.3, 3.3); ax.set_ylim(-3.3, 3.3); ax.set_aspect("equal"); ax.axis("off")
sc_x = axes[0].scatter(*traj[0].T, s=8, color=CA, alpha=0.75, zorder=3)
sc_g = axes[1].scatter(*guess[0].T, s=8, color=NOISE, alpha=0.75, zorder=3)
axes[0].set_title(r"the sample $x_t$", loc="left")
axes[1].set_title(r"what the network thinks $x_0$ is:" "\n"
                  r"$\hat x_0=(x_t-\sqrt{1-\bar\alpha_t}\,\epsilon_\theta)/\sqrt{\bar\alpha_t}$",
                  loc="left", fontsize=10.5)
sup = fig.suptitle("", fontsize=12, y=0.985)
fig.tight_layout(rect=(0, 0, 1, 0.87))


def frame(i):
    k = ks[i]
    sc_x.set_offsets(traj[k])
    sc_g.set_offsets(guess[min(k, T - 1)])
    sup.set_text(f"$t = {T - k}$    ({k} of {T} reverse steps taken)")
    return sc_x, sc_g, sup


anim = FuncAnimation(fig, frame, frames=len(ks))
save_gif(anim, "reverse_chain_2d")

# static fallback: the same two panels at four points along the chain
fig, axes = plt.subplots(2, 4, figsize=(12.4, 6.4))
for j, k in enumerate([0, int(0.70 * T), int(0.93 * T), T]):
    for row, pts, lab in ((0, traj[k], r"$x_t$"), (1, guess[min(k, T - 1)], r"$\hat x_0$")):
        ax = axes[row, j]
        ax.scatter(*data_bg.T, s=2, color=DATA, alpha=0.10, zorder=1)
        ax.scatter(*pts.T, s=6, color=(CA if row == 0 else NOISE), alpha=0.75, zorder=3)
        ax.set_xlim(-3.3, 3.3); ax.set_ylim(-3.3, 3.3); ax.set_aspect("equal"); ax.axis("off")
        if row == 0:
            ax.set_title(f"$t = {T - k}$", loc="left", fontsize=11)
        ax.text(-3.2, -3.1, lab, fontsize=12, color=INK)
fig.tight_layout()
save_png(fig, "reverse_chain_2d")
plt.close("all")

wrote ../figs/diffusion/reverse_chain_2d.gif (1.18 MB)
wrote ../figs/diffusion/reverse_chain_2d.png


## 8. What the sampling chain costs (section "Where Diffusion Falls Short")

The same trained model, sampled with an evenly strided schedule. `sample` with `eta=1` on the
full schedule is the DDPM sampler; the same formula on a strided schedule is how you take fewer,
larger steps. Quality is measured by the energy distance to a real batch of the same size.

In [13]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.gridspec import GridSpec


SHOW = [2, 5, 10, 25, 100, 1000]
KS = [2, 3, 5, 8, 12, 20, 35, 60, 100, 200, 400, 1000]
NS, REPS = 3000, 5

pts, rows = {}, []
for K in sorted(set(KS + SHOW)):
    vals = []
    for rep in range(REPS):
        torch.manual_seed(100 + rep)
        s = sample(net, torch.randn(NS, 2, device=DEVICE), strided(K)).cpu().numpy()
        vals.append(energy_distance(s, moons(NS).cpu().numpy()))
        if rep == 0 and K in SHOW:
            pts[K] = s
    if K in KS:
        rows.append((K, np.mean(vals), np.std(vals)))
ed = np.array(rows)
base = np.array([energy_distance(moons(NS).cpu().numpy(), moons(NS).cpu().numpy()) for _ in range(12)])

fig = plt.figure(figsize=(13.0, 6.6))
gs = GridSpec(2, 1, height_ratios=[1.0, 1.15], hspace=0.3, figure=fig)
gs_top = gs[0].subgridspec(1, len(SHOW), wspace=0.06)

for j, K in enumerate(SHOW):
    ax = fig.add_subplot(gs_top[j])
    ax.scatter(*moons(NS).cpu().numpy().T, s=2, color=DATA, alpha=0.10, zorder=1)
    ax.scatter(*pts[K].T, s=3, color=CA, alpha=0.55, zorder=3)
    ax.set_xlim(-3.3, 3.3); ax.set_ylim(-3.3, 3.3); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_color("#cccccc")
    ax.set_title(f"{K} steps", loc="left", fontsize=11)

ax = fig.add_subplot(gs[1])
ax.errorbar(ed[:, 0], ed[:, 1], yerr=ed[:, 2], fmt="o-", color=CA, lw=2, ms=5,
            ecolor=CA, elinewidth=1, capsize=3)
ax.axhspan(base.min(), base.max(), color=DATA, alpha=0.15, lw=0)
ax.axhline(base.mean(), color=DATA, ls="--", lw=1.4)
ax.text(2.3, 3.4e-4, "what two real batches of the same size score against\n"
        "each other, i.e. the noise floor of this metric", fontsize=9.5, color=DATA)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(1.6, 1500); ax.set_ylim(2.6e-4, 0.9)
ax.set_xlabel("number of reverse steps  =  network evaluations per sample")
ax.set_ylabel("energy distance to the data")
ax.set_title(r"mean $\pm$ s.d. over 5 independent batches of 3000 samples", loc="left", fontsize=11)
ax.annotate("below about 30 steps the\nsamples visibly fall apart", (12, ed[ed[:, 0] == 12][0, 1]),
            xytext=(2.4, 0.0095), fontsize=9.5, color=NOISE,
            arrowprops=dict(arrowstyle="->", color=NOISE, lw=1))
ax.annotate("past that, more steps buy almost nothing,\nbut every one costs another forward pass",
            (750, ed[ed[:, 0] == 1000][0, 1] * 1.4), xytext=(160, 0.035), fontsize=9.5, color=NOISE,
            arrowprops=dict(arrowstyle="->", color=NOISE, lw=1))
fig.text(0.5, 0.955, "the same trained model, sampled with an evenly strided schedule",
         ha="center", fontsize=12.5, color=INK)
save_png(fig, "sampling_cost")
plt.close(fig)
print(ed); print("real-vs-real", base.min(), base.mean(), base.max())

wrote ../figs/diffusion/sampling_cost.png
[[2.00000000e+00 2.63331842e-01 1.26215981e-02]
 [3.00000000e+00 4.20972276e-01 1.26218707e-02]
 [5.00000000e+00 1.70365977e-01 7.82946707e-03]
 [8.00000000e+00 5.85216999e-02 5.30948798e-03]
 [1.20000000e+01 2.14122772e-02 3.34096678e-03]
 [2.00000000e+01 7.85112381e-03 1.91230953e-03]
 [3.50000000e+01 4.03609276e-03 2.23248257e-03]
 [6.00000000e+01 1.78728104e-03 8.90650615e-04]
 [1.00000000e+02 1.88965797e-03 7.24793428e-04]
 [2.00000000e+02 2.03514099e-03 8.63786386e-04]
 [4.00000000e+02 1.27410889e-03 4.22273914e-04]
 [1.00000000e+03 1.56645775e-03 9.37034826e-04]]
real-vs-real 0.00090789794921875 0.0021944443384806314 0.0061457157135009766
